# Week 4 - Model deployment demo
Loads the serialized model (`models/disease_model.joblib`) and calls the FastAPI app in-process with `TestClient`, exactly as an HTTP client would.

In [1]:
import sys, json, joblib
from pathlib import Path
import pandas as pd
WEEK = Path.cwd().parent
sys.path.insert(0, str(WEEK))
model = joblib.load(WEEK / "models" / "disease_model.joblib")
print(type(model).__name__, "->", [name for name, _ in model.steps])
print(json.dumps(json.load(open(WEEK / "models" / "model_metadata.json"))["metrics_holdout"], indent=2))

Pipeline -> ['prep', 'model']
{
  "accuracy": 0.855,
  "precision": 0.6626,
  "recall": 0.7714,
  "f1": 0.7129,
  "roc_auc": 0.8975,
  "cv_f1_mean": 0.6856,
  "cv_f1_std": 0.048,
  "confusion_matrix": [
    [
      405,
      55
    ],
    [
      32,
      108
    ]
  ],
  "train_rows": 2400,
  "test_rows": 600
}


## 1. Direct prediction with the pickled pipeline

In [2]:
sample = pd.DataFrame([{"Age": 70, "Gender": "Male", "BMI": 33, "Blood_Pressure_mmHg": 170, "Cholesterol_mg_dL": 260,
                        "Glucose_mg_dL": 150, "Heart_Rate_bpm": 80, "Smoking": "Yes", "Exercise_Level": "Low"}])
print("P(disease) =", round(model.predict_proba(sample)[0, 1], 4))

P(disease) = 0.9544


## 2. Calling the API endpoints

In [3]:
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app)
print(client.get("/health").json())
r = client.post("/predict", json=sample.iloc[0].to_dict())
print(r.status_code, r.json())

{'status': 'ok', 'model': 'RandomForestClassifier'}
200 {'prediction': 'Disease', 'disease_probability': 0.9544, 'risk_level': 'High'}


/usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 3. Input validation (bad input is rejected with HTTP 422)

In [4]:
bad = dict(sample.iloc[0].to_dict(), Age=-5)
r = client.post("/predict", json=bad)
print(r.status_code, r.json()["detail"][0]["msg"])

422 Input should be greater than or equal to 1
